In [2]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [4]:
from google.colab import files
uploaded = files.upload()

# File upload hone ke baad read karein
import pandas as pd
import io

df = pd.read_csv(io.BytesIO(uploaded['dataset.csv']))
print(df.head())

Saving dataset.csv to dataset.csv
   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again     

In [5]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

# ==========================================
# 1. DATA PREPARATION
# ==========================================
def prepare_data():
    df = pd.read_csv('dataset.csv')
    df = df.drop(columns=['Unnamed: 0'], errors='ignore').dropna()
    df = df.drop_duplicates(subset=['track_name', 'artists'])

    def categorize_mood(row):
        if row['energy'] > 0.8 and row['danceability'] > 0.7: return 'Party'
        if row['instrumentalness'] > 0.5 and row['energy'] < 0.4: return 'Study'
        if row['valence'] >= 0.6 and row['energy'] >= 0.5: return 'Happy'
        if row['valence'] < 0.4 and row['energy'] < 0.4: return 'Sad'
        if row['energy'] >= 0.7: return 'Energetic'
        return 'Calm'

    df['mood'] = df.apply(categorize_mood, axis=1)
    return df

df = prepare_data()
indian_genres = ['indian', 'hindi', 'tamil', 'telugu', 'punjabi', 'bollywood']

# ==========================================
# 2. MEGA FONT SETTINGS (CSS Injection)
# ==========================================
display(HTML("""
    <style>
        /* Labels aur Buttons ka font ekdum bada */
        .widget-label { font-size: 28px !important; color: #1DB954 !important; font-weight: 900 !important; margin-bottom: 10px !important; }
        .jupyter-button { font-size: 22px !important; height: 60px !important; font-weight: bold !important; border-radius: 10px !important; }
        .widget-select select { font-size: 24px !important; height: 200px !important; background: #222 !important; color: white !important; }
        .widget-vbox { background-color: #121212 !important; padding: 40px !important; border-radius: 30px !important; }
    </style>
"""))

# ToggleButtons use kar rhe hain kyunki ye bade aur clear hote hain
mood_w = widgets.ToggleButtons(
    options=['Happy', 'Sad', 'Energetic', 'Calm', 'Party', 'Study'],
    description='🎭 Select Mood:',
    button_style='success', # Green color
    tooltips=['Description of Happy', 'Description of Sad', 'etc'],
)

region_w = widgets.ToggleButtons(
    options=['Bollywood', 'Hollywood'],
    description='🌍 Region:',
    button_style='info',
)

# Singer selection ke liye badi list box
artist_w = widgets.Select(
    options=[],
    description='👤 Singer:',
    layout=widgets.Layout(width='600px', height='250px')
)

# ==========================================
# 3. CORE DASHBOARD LOGIC (Top 10 High Clarity)
# ==========================================
def render_dashboard(*args):
    clear_output(wait=True)
    display(ui_container)

    m, r = mood_w.value, region_w.value
    temp = df[df['mood'] == m]
    if r == 'Bollywood':
        temp = temp[temp['track_genre'].isin(indian_genres)]
    else:
        temp = temp[~temp['track_genre'].isin(indian_genres)]

    artist_list = temp.groupby('artists')['popularity'].sum().sort_values(ascending=False).head(15).index.tolist()
    artist_w.options = artist_list

    sel_artist = artist_w.value
    if not sel_artist: return

    final_songs = temp[temp['artists'] == sel_artist].sort_values(by='popularity', ascending=False).head(10)
    color = {'Happy': '#FFD700', 'Sad': '#1E90FF', 'Energetic': '#FF4500', 'Calm': '#00FA9A', 'Party': '#FF1493', 'Study': '#A020F0'}.get(m, '#1DB954')

    html_output = f"""
    <div style="background: #000; padding: 40px; border-radius: 30px; font-family: Arial; border: 4px solid {color}; margin-top: 40px; max-width: 1000px;">
        <div style="margin-bottom: 40px; border-left: 15px solid {color}; padding-left: 30px;">
            <h1 style="color: white; font-size: 60px; margin: 0; font-weight: 900;">{sel_artist}</h1>
            <p style="color: {color}; font-size: 32px; font-weight: bold; margin-top: 15px; letter-spacing: 2px;">{m.upper()} TOP 10</p>
        </div>
        <div style="display: flex; flex-direction: column; gap: 25px;">
    """

    for _, row in final_songs.iterrows():
        track_id = row['track_id']
        embed = f'<iframe src="https://open.spotify.com/embed/track/{track_id}" width="100%" height="100" frameBorder="0" allowtransparency="true" allow="encrypted-media" style="border-radius:20px;"></iframe>'

        html_output += f"""
        <div style="background: #181818; border-radius: 25px; padding: 30px; display: flex; align-items: center; gap: 35px; border: 1px solid #333; box-shadow: 0 15px 35px rgba(0,0,0,0.7);">
            <img src="https://picsum.photos/seed/{track_id}/200/200" style="width: 150px; height: 150px; border-radius: 20px; border: 2px solid {color}33;">
            <div style="flex-grow: 1;">
                <h3 style="color: white; font-size: 34px; margin: 0 0 15px 0; font-weight: 900; line-height: 1.2;">{row['track_name']}</h3>
                {embed}
            </div>
        </div>
        """

    html_output += "</div></div>"
    display(HTML(html_output))

# ==========================================
# 4. RUN APP
# ==========================================
mood_w.observe(render_dashboard, 'value')
region_w.observe(render_dashboard, 'value')
artist_w.observe(render_dashboard, 'value')

ui_container = widgets.VBox([
    widgets.HTML("<h1 style='color: #1DB954; font-size: 55px; margin: 20px; font-weight: 900; font-family: Arial;'>Aryan Bhai's Pro Player</h1>"),
    mood_w,
    region_w,
    artist_w
], layout=widgets.Layout(padding='20px'))

display(ui_container)
render_dashboard()